<a href="https://colab.research.google.com/github/springboardmentor2468a-lab/Projects_2/blob/Anish-kumar/MODEL%20PREDICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Imports* All

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


**Remove_outliers**

In [6]:
def remove_outliers_iqr(df):
    numeric_cols = df.select_dtypes(include='number').columns
    Q1 = df[numeric_cols].quantile(0.25)
    Q3 = df[numeric_cols].quantile(0.75)
    IQR = Q3 - Q1

    return df[~(
        (df[numeric_cols] < (Q1 - 1.5 * IQR)) |
        (df[numeric_cols] > (Q3 + 1.5 * IQR))
    ).any(axis=1)]


GridSearch **Evaluate**

In [7]:
def evaluate_grid(grid, X_train, X_test, y_train, y_test):
    model = grid.best_estimator_
    return (
        r2_score(y_train, model.predict(X_train)),
        r2_score(y_test, model.predict(X_test))
    )


HOUR **DATASET**

In [8]:
# ---------- Load HOUR data ----------
hour = pd.read_csv("hour.csv")

# Convert date
hour['dteday'] = pd.to_datetime(hour['dteday'])

# Drop unwanted columns (HOUR DATA CLEANING)
cols_to_drop = ['instant', 'dteday', 'yr', 'casual', 'registered']
hour.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Remove outliers (HOUR DATA)
hour = remove_outliers_iqr(hour)


Train–Test Split

In [9]:
X_hour = hour.drop('cnt', axis=1)
y_hour = hour['cnt']

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_hour, y_hour, test_size=0.2, random_state=42
)


Linear Regression with Scaling

In [10]:
scaler_hour = StandardScaler()
Xh_train_scaled = scaler_hour.fit_transform(Xh_train)
Xh_test_scaled  = scaler_hour.transform(Xh_test)

lr_hour = LinearRegression()
lr_hour.fit(Xh_train_scaled, yh_train)

print("HOUR DATA - Linear Regression")
print("Train R2:", r2_score(yh_train, lr_hour.predict(Xh_train_scaled)))
print("Test  R2:", r2_score(yh_test, lr_hour.predict(Xh_test_scaled)))


HOUR DATA - Linear Regression
Train R2: 0.36269497597545386
Test  R2: 0.35370564344651856


Decision Tree

In [11]:
dt_hour = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_hour.fit(Xh_train, yh_train)

print("\nHOUR DATA - Decision Tree")
print("Train R2:", r2_score(yh_train, dt_hour.predict(Xh_train)))
print("Test  R2:", r2_score(yh_test, dt_hour.predict(Xh_test)))



HOUR DATA - Decision Tree
Train R2: 0.8501356611488479
Test  R2: 0.7958862057796872


Random Forest

In [12]:
rf_hour = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42
)

rf_hour.fit(Xh_train, yh_train)

print("\nHOUR DATA - Random Forest")
print("Train R2:", r2_score(yh_train, rf_hour.predict(Xh_train)))
print("Test  R2:", r2_score(yh_test, rf_hour.predict(Xh_test)))



HOUR DATA - Random Forest
Train R2: 0.9571318238161559
Test  R2: 0.856451895880521


**DAY DATASET**

In [13]:
# ---------- Load DAY data ----------
day = pd.read_csv("day.csv")

# Convert date
day['dteday'] = pd.to_datetime(day['dteday'])

# Drop unwanted columns (DAY DATA CLEANING)
day.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Remove outliers (DAY DATA)
day = remove_outliers_iqr(day)


Train–Test Split

In [14]:
X_day = day.drop('cnt', axis=1)
y_day = day['cnt']

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_day, y_day, test_size=0.2, random_state=42
)


Linear Regression

In [15]:
scaler_day = StandardScaler()
Xd_train_scaled = scaler_day.fit_transform(Xd_train)
Xd_test_scaled  = scaler_day.transform(Xd_test)

lr_day = LinearRegression()
lr_day.fit(Xd_train_scaled, yd_train)

print("\nDAY DATA - Linear Regression")
print("Train R2:", r2_score(yd_train, lr_day.predict(Xd_train_scaled)))
print("Test  R2:", r2_score(yd_test, lr_day.predict(Xd_test_scaled)))



DAY DATA - Linear Regression
Train R2: 0.5300657560744655
Test  R2: 0.4651527230669633


Random Forest

In [16]:
rf_day = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42
)

rf_day.fit(Xd_train, yd_train)

print("\nDAY DATA - Random Forest")
print("Train R2:", r2_score(yd_train, rf_day.predict(Xd_train)))
print("Test  R2:", r2_score(yd_test, rf_day.predict(Xd_test)))



DAY DATA - Random Forest
Train R2: 0.9488957010860649
Test  R2: 0.564726127360927


**CHOOSE BEST MODEL**

In [18]:
best_hour_model = rf_hour  # HOUR dataset best model
best_day_model  = rf_day                    # DAY dataset best model


FUTURE PREDICTION

In [20]:
def predict_future(model, future_input_df):
    """
    model: trained ML model
    future_input_df: DataFrame with SAME features as training data
    """
    return model.predict(future_input_df)


**HOUR FUTURE PREDICTIONS**

In [21]:
hour_plus_1_input = pd.DataFrame([{
    'season': 2,
    'mnth': 7,
    'hr': 15,
    'holiday': 0,
    'weekday': 3,
    'workingday': 1,
    'weathersit': 1,
    'temp': 0.32,
    'atemp': 0.30,
    'hum': 0.55,
    'windspeed': 0.12
}])

hour_plus_6_input = pd.DataFrame([{
    'season': 2,
    'mnth': 7,
    'hr': 21,
    'holiday': 0,
    'weekday': 3,
    'workingday': 1,
    'weathersit': 1,
    'temp': 0.30,
    'atemp': 0.28,
    'hum': 0.60,
    'windspeed': 0.15
}])



Predictions

In [22]:
cnt_hour_plus_1 = predict_future(best_hour_model, hour_plus_1_input)
cnt_hour_plus_6 = predict_future(best_hour_model, hour_plus_6_input)

print("Predicted cnt (hour+1):", cnt_hour_plus_1[0])
print("Predicted cnt (hour+6):", cnt_hour_plus_6[0])


Predicted cnt (hour+1): 132.76159361633708
Predicted cnt (hour+6): 75.98253701192489


**DAY FUTURE PREDICTIONS**

In [24]:
day_plus_1_input = pd.DataFrame([{
    'season': 2,
    'mnth': 7,
    'holiday': 0,
    'weekday': 4,
    'workingday': 1,
    'weathersit': 1,
    'temp': 0.33,
    'atemp': 0.31,
    'hum': 0.52,
    'windspeed': 0.13
}])
day_plus_6_input = pd.DataFrame([{
    'season': 2,
    'mnth': 7,
    'holiday': 0,
    'weekday': 2,
    'workingday': 1,
    'weathersit': 2,
    'temp': 0.29,
    'atemp': 0.27,
    'hum': 0.60,
    'windspeed': 0.18
}])



In [25]:
cnt_day_plus_1 = predict_future(best_day_model, day_plus_1_input)
cnt_day_plus_6 = predict_future(best_day_model, day_plus_6_input)

print("Predicted cnt (day+1):", cnt_day_plus_1[0])
print("Predicted cnt (day+6):", cnt_day_plus_6[0])


Predicted cnt (day+1): 3875.908676470588
Predicted cnt (day+6): 3207.601333333333
